# 02 — Feature Engineering
Resample mixed-frequency data, build lag features, rolling statistics, and save the processed feature matrix.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.features import (
    resample_to_monthly,
    make_stationary,
    make_lag_features,
    make_rolling_features,
    build_feature_matrix,
)

print('Imports OK')

In [ ]:
df_raw = pd.read_csv('../data/raw/macro_raw.csv', index_col='date', parse_dates=True)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

## Step 1 — Resample to monthly

In [ ]:
df_monthly = resample_to_monthly(df_raw)
print(f'Monthly shape: {df_monthly.shape}')
print(f'Date range: {df_monthly.index[0].date()} → {df_monthly.index[-1].date()}')
df_monthly.isnull().sum()

## Step 2 — Stationarity transform

In [ ]:
df_stat = make_stationary(df_monthly, method='pct_change')

# Visualize before/after for GDP
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
df_monthly['gdp'].dropna().plot(ax=ax1, title='GDP — Level (non-stationary)')
df_stat['gdp'].dropna().plot(ax=ax2, title='GDP — % Change (stationary)', color='tomato')
plt.tight_layout()
plt.savefig('../outputs/figures/gdp_stationarity.png', dpi=150)
plt.show()

## Step 3 — Lag + rolling features

In [ ]:
indicator_cols = [c for c in df_stat.columns if c not in ['gdp', 'cpi', 'unemployment']]

df_feat = make_lag_features(df_stat, lags=[1, 2, 3, 4, 6, 12], cols=indicator_cols)
df_feat = make_rolling_features(df_feat, windows=[3, 6, 12], cols=indicator_cols)

print(f'Feature matrix shape: {df_feat.shape}')
print(f'Total features: {df_feat.shape[1]}')
df_feat.head(3)

## Step 4 — Build X, y for each target and horizon

In [ ]:
targets = ['gdp', 'cpi', 'unemployment']
horizons = [1, 2, 4]

datasets = {}
for target in targets:
    for h in horizons:
        X, y = build_feature_matrix(df_raw, target_col=target, forecast_horizon=h)
        datasets[(target, h)] = (X, y)
        print(f'{target} h={h}: X={X.shape}, y={y.shape}, NaN in y={y.isna().sum()}')

## Step 5 — Save processed data

In [ ]:
# Save the monthly stationary frame as the base processed file
df_stat.to_csv('../data/processed/macro_monthly_stationary.csv')
df_feat.to_csv('../data/processed/macro_features.csv')
print('Saved to data/processed/')